# Lab 8 - Manual CPA Attack

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://learn.ChipWhisperer.io/courses/sca101). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the last lab, we introduced AES and showed how a technique called Correlation Power Analysis can be used to recover secret keys from devices running AES. We then used ChipWhisperer Analyzer to perform a CPA attack and recover an AES key.*

*In this lab, we'll be reenforcing our understanding of CPA attacks by doing an attack without using Analyzer. To simplify the attack, we'll be using the SubBytes location we learned in the previous lab to attack only that point.*

**LEARNING OUTCOMES:**

* Calculating correlation
* Using correlation to rank AES Key Guesses

## Prerequisites

Hold up! Before you continue, check you've done the following tutorials:

* ☑ [Lab 7 - Automated Correlation Power Analysis Attack](../Lab%207%20-%20Automated%20Correlation%20Power%20Analysis%20Attack/Lab.ipynb).

## Intro

In our previous lab, we learned how we can use the linear relationship between the Hamming weight of data and power consumption to recover secret information from our SAM4S. We did this by making the following assumptions:

1. If we correctly guess secret data combined with varying known data and the sample point of an operation acting on that data, the Hamming weight of that guess will have a linear relationship with the power consumption
2. If we get either the result of the operation, or the sample point wrong, the Hamming weight of the guess and the power consumption will not have a linear relationship.

To measure the linearity of the relationship, we used the Pearson Correlation Coefficient. The absolute value of this correlation varies from 0 to 1, with 0 being non-linear and 1 being perfectly linear. We also saw how it was important that there was some sort of non-linear operation between the combination of the known and secret data, and the output of the operation that we're trying to measure.

This was all done with a focus on AES, which is a symmetric encryption algorithm. By selecting the output of SubBytes as our operation and using ChipWhisperer's Analyzer library, we were able to recover a full 16-byte AES key in under 50 traces.

The goal for this lab is to cut Analyzer out entirely and use a manual correlation calculation to find the correct key. To make this a bit simpler, we're going to use the location of the SubBytes operation from the previous lab to limit our attack to a single point.

## Connection and Capture

Setup and capture are the same as in the previous lab

In [ ]:
%%bash
cd ../../firmware/mcu/simpleserial-aes
make PLATFORM=CWHUSKY CRYPTO_TARGET=TINYAES128C SS_VER=SS_VER_1_1 -j

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-aes/simpleserial-aes-CWHUSKY.hex")

Capture your traces like in the previous lab below:

## AES Recap

We went over AES in the previous lab, but here's a very quick review:

1. 16-bytes of plaintext, key, ciphertext, and state. The state is orgranized into a 4x4 matrix
2. 4 operations: AddRoundKey, SubBytes, ShiftRows, and MixColumns
3. ARK is `output = state ^ key`
4. SubBytes is `output = sbox[state]`

## Calculating Correlation

The Pearson correlation coefficient is a measurement of the linearity of the relationship between two datasets. Its absolute value varies between
0 and 1, with 0 being totally non-linear and 1 being perfectly linear. Its sign tells you the sign of the slope of the relationship.

It can be calculated as follows:

$r = \frac{cov(X,Y)}{\sigma_x \sigma_y}$

$cov(X,Y) = \frac{1}{N} \sum_{n=1}^{N} [(Y_n - \bar{Y})(X_n - \bar{X})] = \frac{1}{N}((Y-\bar{Y}) \cdot (X - \bar{X}))$

$\sigma_x = \sqrt{\sum_{n=1}^{N}(X_n - \bar{X})^2}$

$\bar{X} = \frac{1}{N}\sum_{n=1}^{N}X_n$

**Create a function to calculate the correlation between two datasets. Do this in anyway you like. Numpy has a function for calculating correlation, but you may find it easier to calculate via the dot product, mean, and standard deviation.**

I recommend returning the **absolute value of the correlation**, as we don't care about the sign of the slope of the relationship.

## Calculating Leakage Guesses

**The last step before actually running the attack is to calculate the hypothetical leakages. Below you'll find everything you need to do this calculation.**

In [ ]:
import numpy as np

def generate_hw_table():
    ret = []
    for i in range(256):
        ret.append(bin(i).count('1'))
    return np.array(ret, dtype=np.uint8)

sbox = np.array([
# 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f 
0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
], dtype=np.uint8)

hw_table = generate_hw_table()

## Running Attack

Now you need to find the kguess with the highest correlation. This will be very similar to what we did for the password guess lab.

For reference, one possible solution to that lab was:

```python
for c in tqdm('abcdefghijklmnopqrstuvwxyz0123456789'): 
    trace = cap_pass_trace("h" + c + "\n")[0]
    diff = np.sum(np.abs(trace - ref_trace))
    if diff > biggest_diff:
        biggest_diff = diff
        biggest_diff_c = c
        print("New best guess {} ({})".format(c, diff))
```

So you need something like:

```python
for kguess in kguesses:
    hyp = subbytes_leakage(pt[:, 0], kguess)
    r = corr(hyp, traces[:, sb_loc])
    if r > biggest_corr:
        #update guess
        #update biggest_corr
```

**Use your correlation function and leakage calculation function to do a CPA attack:**

## Extending the Attack

We simplified the attack by just using a single location and a single byte. We won't go through it here, but to attack the entire trace, you just need to repeat the correlation calculation for each point sample point in our power trace. Feel free to try this, but you may want to limit your range on the power trace, as the way we're calculating correlation is fairly slow.

Similarly, you can extend the attack to the entire key by repeating the attack with the other bytes of the plaintext. Note that the SubBytes location will be different for each byte, so you'll need to modify the trace location, or search along the power trace.

## Conclusion

Congratulations! You've now recovered part of an AES key on your own. This lab should make it even more clear how our attack from the previous lab is working. You probably also saw how much faster Analyzer is than a simple calculation like this, especially if you attempted the attack over the entire power trace.

## Appendix A: Online Correlation Calculation

Currently, the loop needs to go through all the traces before it can return a correlation. This isn't too bad for a short attack, for a much longer one (think 10k+ traces) we won't get any feedback from the attack until it's finished. Also, if we didn't capture enough traces for the attack, the entire analysis calculation needs to be repeated! Instead of using the original correlation equation, we can instead use an equivalent "online" version that can be easily updated with more traces: $$r_{i,j} = \frac{N\sum_{n=1}^{N}h_{n,i}t_{n,j}-\sum_{n=1}^{N}h_{n,i}\sum_{n=1}^{N}t_{n,j}}{\sqrt{((\sum_{n=1}^Nh_{n,i})^2-N\sum_{n=1}^Nh_{n,i}^2)-((\sum_{n=1}^Nt_{n,j})^2-N\sum_{n=1}^Nt_{n,j}^2)}}$$

| **Equation** | **Python Variable** | **Value**  | 
|--------------|---------------------|------------|
|  n           |       tnum          | trace number |
|  i           |       kguess        | subkey guess |
| j | j index trace point | sample point in trace |
| h | hypint | guess for power consumption (`HW(sbox(pt ^ key))`)| 
| t | traces | traces | 

or, more abstractly

$$r_{i,j} = \frac{Nh_i\cdot t_j-\bar{h_i}\bar{t_j}}{\sqrt{(\bar{h_i}^2-N\bar{h_i^2})-(\bar{t_j}^2-N\bar{t_j^2})}}$$
where

* $N$ is the number of traces
* $h_i\cdot t_j$ is the dot product between leakage guesses and traces
* $\bar{h_i}$ is the mean of the leakage guesses
* $\bar{h_i^2}$ is the mean of the square of the leakage guesses
* $\bar{h_i}^2$ is the square of the mean of the leakage guesses

Each of these values can be stored and updated as more traces are added. This is how ChipWhisperer Analyzer does the calculations and is
how we can show the updating table as the attack is being run! If you want, try calculating this version of the correlation for your correct key guess.

In [ ]:
scope.dis()
target.dis()

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2026. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>